In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [14]:
url = "https://www.nykaa.com/search/result/?q=healthcare&root=search&searchType=Manual&sourcepage=home"
headers = {
    "User-Agent": "Mozilla/5.0"
}


In [15]:
response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)
print("Length:", len(response.content))

Status Code: 200
Length: 443669


In [8]:
soup = BeautifulSoup(response.content, "html.parser")

titles = soup.find_all("h2", class_="css-xrzmfa")

print("Number of products:", len(titles))

Number of products: 20


In [9]:
# Empty list to store product data
products = []

for product in titles[:20]:

    # Product Name
    product_name = product.get_text(strip=True)

    # Product card text
    parent = product.parent
    text = parent.get_text(" ", strip=True)

    # MRP
    mrp_match = re.search(r"Regular price ₹(\d+)", text)

    if mrp_match:
        mrp = mrp_match.group(1)
    else:
        mrp = "N/A"

    # Selling Price
    price_match = re.search(r"Discounted price ₹(\d+)", text)

    if price_match:
        price = price_match.group(1)
    else:
        price = "N/A"

    # Discount
    discount_match = re.search(r"(\d+)% Off", text)

    if discount_match:
        discount = discount_match.group(1) + "%"
    else:
        discount = "N/A"

    # Brand
    brand_match = re.match(r"(\S+)", product_name)

    if brand_match:
        brand = brand_match.group(1)
    else:
        brand = "N/A"

    # Rating
    rating = "N/A"

    current = product

    for i in range(5):

        if current:

            current_text = current.get_text(" ", strip=True)

            rating_match = re.search(r"\b[0-5]\.\d\b", current_text)

            if rating_match:
                rating = rating_match.group(0)
                break

            current = current.parent

    # Reviews
    reviews_match = re.search(r"\(\s*(\d+)\s*\)", text)

    if reviews_match:
        reviews = reviews_match.group(1)
    else:
        reviews = "N/A"

    # Product URL
    link = product.find_parent("a", href=True)

    if link:
        product_url = "https://www.nykaa.com" + link["href"]
    else:
        product_url = "N/A"

    # Category
    category = "Healthcare"

    # Store product data
    products.append({
        "Product Name": product_name,
        "Brand": brand,
        "MRP": mrp,
        "Selling Price": price,
        "Discount": discount,
        "Rating": rating,
        "Reviews": reviews,
        "Product URL": product_url,
        "Category": category
    })




In [10]:
df = pd.DataFrame(products)
df

,Product Name,Brand,MRP,Selling Price,Discount,Rating,Reviews,Product URL,Category
0,hoop Magnesium Sleep Body Lotion Topical Magne...,hoop,680,520,24%,N/A,366,https://www.nykaa.com/hoop-magnesium-sleep-bod...,Healthcare
1,"Be Bodywise Biotin Hair Gummies - Zinc, Fibre,...",Be,999,849,15%,N/A,12318,https://www.nykaa.com/be-bodywise-biotin-hair-...,Healthcare
2,"Be Bodywise Biotin Hair Gummies - Zinc, Fibre,...",Be,N/A,N/A,N/A,N/A,12318,https://www.nykaa.com/be-bodywise-biotin-hair-...,Healthcare
3,HealthKart Hk Vitals Skin Radiance Collagen Su...,HealthKart,975,909,7%,N/A,10214,https://www.nykaa.com/healthkart-hk-vitals-ski...,Healthcare
4,Oziva Advanced Hair Growth ActivesReduces Thin...,Oziva,2099,1849,12%,N/A,280,https://www.nykaa.com/oziva-advanced-hair-grow...,Healthcare
5,Plix Apple Cider Vinegar Effervescent Tablets ...,Plix,1245,1073,14%,N/A,740,https://www.nykaa.com/plix-world-s-first-acv-j...,Healthcare
6,HealthKart Hk Vitals Multivitamin + Fish Oil,HealthKart,669,489,27%,N/A,85,https://www.nykaa.com/healthkart-hk-vitals-mul...,Healthcare
7,Be Bodywise Collagen Builder Gummies Vitamins ...,Be,999,449,55%,N/A,1380,https://www.nykaa.com/be-bodywise-collagen-ski...,Healthcare
8,What's Up Wellness Glutathione Gummies with Vi...,What's,1599,1149,28%,N/A,109,https://www.nykaa.com/what-s-up-wellness-gluta...,Healthcare
9,Kapiva Wild Amla Juice (Cold Pressed)Suitable ...,Kapiva,280,233,17%,N/A,1214,https://www.nykaa.com/kapiva-ayurveda-amla-jui...,Healthcare


In [11]:
df.to_csv("nykaa_healthcare_products.csv", index=False)

print("CSV file created successfully!")
print("Total products saved:", len(df))

CSV file created successfully!
Total products saved: 20
